# Statistics from First Principles
### with an AI tutor powered by Groq

This notebook teaches the same twelve ideas as the visual deck — but here you'll compute
every formula **by hand in Python**, then check it against `numpy` / `scipy`, then ask an
AI tutor to explain *why* the formula is shaped the way it is and where it shows up in the
real world.

**How each section is organized:**
1. 🧮 **From scratch** — the formula, implemented with plain loops/arithmetic so nothing is hidden in a library call.
2. ✅ **Check with a library** — the same number from `numpy`/`scipy`/`pandas`, to build trust that the from-scratch version is right.
3. 📊 **A picture** — a quick plot, because a number alone rarely builds intuition.
4. 🤖 **Ask the AI tutor** — a Groq-backed LLM explains the *why* and gives a real-world example, in its own words.
5. ✏️ **Your turn** — a small blank exercise to try the same calculation on your own numbers.

**Table of contents**
1. Mean · 2. Median · 3. Mode · 4. Variance · 5. Standard Deviation · 6. Covariance ·
7. Correlation · 8. Percentiles · 9. Confidence Interval · 10. Hypothesis Testing ·
11. p-value · 12. ANOVA

## Setup

Install the packages we need (safe to re-run; already-installed packages are skipped),
then set up the Groq API key.

> ⚠️ **A note on API keys:** never commit a real API key to source control or share a
> notebook that has one hardcoded in it. The pattern below only *sets* the key if it
> isn't already in your environment, and in your own projects you should prefer loading
> it from a `.env` file (`python-dotenv`) or your shell environment instead of hardcoding
> it in a cell. If the key below is a placeholder, swap in your own from
> [console.groq.com/keys](https://console.groq.com/keys).

In [ ]:
# If a package is missing, uncomment the next line and run this cell.
# %pip install -q groq numpy pandas scipy matplotlib

In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)               # reproducible "random" examples throughout
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

pd.set_option("display.precision", 4)
print("Libraries loaded.")

### The AI tutor helper

One small wrapper around the Groq chat API. If the API key isn't valid or there's no
network access, it fails gracefully and tells you so instead of crashing the notebook —
every other cell in this notebook works completely fine without it; the AI tutor is a
bonus layer on top of the math, not a requirement for it.

In [ ]:
from groq import Groq

_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Groq's current fast general-purpose model (as of mid-2026).
# Swap this for any other model id from https://console.groq.com/docs/models
GROQ_MODEL = "openai/gpt-oss-20b"

TUTOR_SYSTEM_PROMPT = (
    "You are a patient statistics tutor in the style of Richard Feynman: prefer plain "
    "language and concrete intuition over jargon, explain WHY a formula is shaped the way "
    "it is (not just what it computes), and always end with one short, specific real-world "
    "example. Keep answers under 180 words."
)

def ask_ai(question, system=TUTOR_SYSTEM_PROMPT, model=GROQ_MODEL, temperature=0.4):
    """Ask the Groq-hosted AI tutor a question and print the answer."""
    try:
        resp = _client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": question},
            ],
        )
        answer = resp.choices[0].message.content
        print(answer)
        return answer
    except Exception as e:
        print(
            "⚠️  Couldn't reach the AI tutor (this is optional — the math above already "
            f"stands on its own).\n    Reason: {e}"
        )
        return None

In [ ]:
# Quick smoke test — comment this out if you'd rather save API calls for later.
ask_ai("In one or two sentences, what will this notebook teach me?")

---
## 1. Mean

**Definition:** the arithmetic average — sum of the values divided by how many there are.

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad\qquad \mu = \frac{1}{N}\sum_{i=1}^{N} x_i$$

$\bar{x}$ (sample mean) and $\mu$ (population mean) are the *same formula* — which symbol
you use just depends on whether your data is the whole population or a sample from it.

In [ ]:
data = [2, 3, 5, 7, 8]

# 1. From scratch
total = 0
for x in data:
    total += x
mean_from_scratch = total / len(data)
print(f"sum = {total}, n = {len(data)}, mean = {mean_from_scratch}")

In [ ]:
# 2. Check with a library
mean_numpy = np.mean(data)
mean_pandas = pd.Series(data).mean()
print("numpy :", mean_numpy)
print("pandas:", mean_pandas)
assert math.isclose(mean_from_scratch, mean_numpy)

In [ ]:
# 3. A picture — the mean as a balance point
fig, ax = plt.subplots(figsize=(7, 1.8))
ax.scatter(data, [0]*len(data), s=140, zorder=3, color="#4a7c9e")
ax.axvline(mean_from_scratch, color="#c98a4b", linestyle="--", label=f"mean = {mean_from_scratch}")
ax.set_yticks([])
ax.set_title("The mean is where the data balances")
ax.legend()
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Explain why the mean is the value that minimizes the sum of squared deviations, "
    "using the data [2, 3, 5, 7, 8] with mean 5 as a concrete example."
)

✏️ **Your turn:** replace `data` below with your own numbers (e.g. your last 5 grocery
bills) and compute the mean from scratch — no `np.mean`, just a loop or `sum()/len()`.

In [ ]:
data_yours = []  # <- put your own numbers here
# mean_yours = ...

---
## 2. Median

**Definition:** the middle value once the data is sorted — the 50th percentile.

$$
\text{median} =
\begin{cases}
x_{\left(\frac{n+1}{2}\right)} & n \text{ odd} \\[4pt]
\dfrac{x_{(n/2)} + x_{(n/2+1)}}{2} & n \text{ even}
\end{cases}
$$

Unlike the mean, the median is **robust** — moving the largest value arbitrarily far away
doesn't move the median at all.

In [ ]:
data = [7, 2, 9, 4, 5]

# 1. From scratch
sorted_data = sorted(data)
n = len(sorted_data)
if n % 2 == 1:
    median_from_scratch = sorted_data[n // 2]
else:
    median_from_scratch = (sorted_data[n // 2 - 1] + sorted_data[n // 2]) / 2

print("sorted:", sorted_data)
print("median:", median_from_scratch)

In [ ]:
# 2. Check with a library
print("numpy :", np.median(data))
print("pandas:", pd.Series(data).median())
assert math.isclose(median_from_scratch, np.median(data))

In [ ]:
# 3. A picture — mean vs. median under an outlier
normal_data = [7, 2, 9, 4, 5]
with_outlier = normal_data + [500]     # one huge outlier

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, d, title in zip(axes, [normal_data, with_outlier], ["No outlier", "With one outlier (500)"]):
    ax.scatter(d, [0]*len(d), s=100, color="#4a7c9e")
    ax.axvline(np.mean(d), color="#b3728a", linestyle="--", label=f"mean={np.mean(d):.1f}")
    ax.axvline(np.median(d), color="#c98a4b", linestyle=":", label=f"median={np.median(d):.1f}")
    ax.set_yticks([]); ax.set_title(title); ax.legend()
plt.suptitle("The median barely moves; the mean gets dragged")
plt.tight_layout(); plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Why does the median minimize the sum of ABSOLUTE deviations instead of squared "
    "deviations, and why does that make it resistant to outliers? Keep it intuitive."
)

✏️ **Your turn:** add one absurdly large value to `data` above and compare how much the
mean moves versus how much the median moves.

In [ ]:
# your experiment here

---
## 3. Mode

**Definition:** the most frequently occurring value in the data. It's the only central
tendency measure that also works for **categorical** data (you can't average colors, but
you can find the most common one).

In [ ]:
data = [2, 3, 3, 3, 5, 5, 7]

# 1. From scratch
counts = {}
for x in data:
    counts[x] = counts.get(x, 0) + 1

mode_from_scratch = max(counts, key=counts.get)
print("frequency table:", counts)
print("mode:", mode_from_scratch)

In [ ]:
# 2. Check with a library
print("scipy :", stats.mode(data, keepdims=False))
print("pandas:", pd.Series(data).mode().tolist())   # pandas returns ALL modes (ties included)

In [ ]:
# 3. A picture
fig, ax = plt.subplots(figsize=(6, 3.5))
values = list(counts.keys())
freqs = list(counts.values())
colors = ["#7fa38a" if v == mode_from_scratch else "#4a7c9e" for v in values]
ax.bar(values, freqs, color=colors, width=0.5)
ax.set_xlabel("value"); ax.set_ylabel("frequency")
ax.set_title(f"mode = {mode_from_scratch} (tallest bar)")
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Give a real-world example where the mode is clearly the right statistic to report "
    "and the mean would be nonsensical or even impossible to use."
)

✏️ **Your turn:** find the mode of a categorical list, e.g.
`["red", "blue", "blue", "green", "blue", "red"]` — notice `np.mean` would fail here, but
your frequency-counting approach still works.

In [ ]:
colors = ["red", "blue", "blue", "green", "blue", "red"]
# your mode calculation here

---
## 4. Variance

**Definition:** the average *squared* deviation from the mean — one number summarizing spread.

$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2 \qquad\qquad s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2$$

The population formula divides by $N$. The **sample** formula divides by $n-1$ instead of
$n$ — this is **Bessel's correction**. Why? $\bar x$ is computed *from* the same sample, and
is, by construction, the value that minimizes $\sum(x_i-\bar x)^2$ for that sample — so this
sum is systematically a little smaller than if you'd used the true (unknown) $\mu$.
Dividing by $n-1$ corrects for that shrinkage, on average.

In [ ]:
data = [2, 3, 5, 7, 8]
mean = sum(data) / len(data)

# 1. From scratch — population variance (divide by n)
squared_devs = [(x - mean) ** 2 for x in data]
population_variance = sum(squared_devs) / len(data)

# ...and sample variance (divide by n-1)
sample_variance = sum(squared_devs) / (len(data) - 1)

print("deviations       :", [round(x - mean, 2) for x in data])
print("squared deviations:", squared_devs)
print("population variance (÷n)  :", population_variance)
print("sample variance     (÷n-1):", sample_variance)

In [ ]:
# 2. Check with a library — numpy defaults to population variance (ddof=0);
#    pass ddof=1 to get the sample variance instead.
print("numpy population variance:", np.var(data, ddof=0))
print("numpy sample variance    :", np.var(data, ddof=1))
print("pandas (sample by default):", pd.Series(data).var())
assert math.isclose(population_variance, np.var(data, ddof=0))
assert math.isclose(sample_variance, np.var(data, ddof=1))

In [ ]:
# 3. A picture — deviations, squared, as literal squares
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axhline(0, color="#666", lw=1)
ax.axvline(mean, color="#c98a4b", ls="--", label=f"mean={mean}")
for x in data:
    d = x - mean
    side = abs(d)
    # draw the "deviation squared" as an actual square, sitting on the axis
    x0 = min(x, mean)
    rect = plt.Rectangle((x0, 0), side, side, alpha=0.25, color="#b3728a")
    ax.add_patch(rect)
    ax.scatter([x], [0], color="#4a7c9e", zorder=3, s=80)
    ax.text(x0 + side/2, side + 0.15, f"{d:+.0f}² = {d**2:.0f}", ha="center", fontsize=9)
ax.set_xlim(0, 10); ax.set_ylim(-0.5, 5)
ax.set_title("Each squared deviation, drawn as a literal square of area (x−mean)²")
ax.legend()
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Explain, with intuition rather than proof, why sample variance divides by (n-1) "
    "instead of n — what is Bessel's correction actually correcting for?"
)

✏️ **Your turn:** compute variance for `[10, 12, 23, 23, 16, 23, 21, 16]` from scratch,
then confirm with `np.var`.

In [ ]:
data_yours = [10, 12, 23, 23, 16, 23, 21, 16]
# your variance calculation here

---
## 5. Standard Deviation

**Definition:** $\sigma = \sqrt{\sigma^2}$ — variance brought back into the *original units*.
Variance of a dataset in dollars is in dollars²; nobody thinks in dollars². The square root
undoes exactly that, and only that.

**The empirical rule** (for roughly bell-shaped data): about 68% of values fall within
1σ of the mean, 95% within 2σ, and 99.7% within 3σ.

In [ ]:
data = [2, 3, 5, 7, 8]
variance = np.var(data, ddof=0)

# 1. From scratch
std_from_scratch = variance ** 0.5
print(f"variance = {variance}")
print(f"std dev  = sqrt({variance}) = {std_from_scratch:.4f}")

In [ ]:
# 2. Check with a library
print("numpy :", np.std(data, ddof=0))
print("pandas (sample std):", pd.Series(data).std())
assert math.isclose(std_from_scratch, np.std(data, ddof=0))

In [ ]:
# 3. A picture — the empirical rule on a much larger, actually-normal sample
sample = np.random.normal(loc=100, scale=15, size=5000)   # e.g. IQ-like scores
mu, sigma = sample.mean(), sample.std()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample, bins=60, density=True, color="#4a7c9e", alpha=0.6)
xs = np.linspace(sample.min(), sample.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, mu, sigma), color="#111", lw=2)
for k, alpha in zip([1, 2, 3], [0.30, 0.18, 0.08]):
    ax.axvspan(mu - k*sigma, mu + k*sigma, color="#c98a4b", alpha=alpha)
ax.set_title(f"mean={mu:.1f}, σ={sigma:.1f} — shaded bands are ±1σ, ±2σ, ±3σ")
plt.show()

within_1 = np.mean(np.abs(sample - mu) <= sigma)
within_2 = np.mean(np.abs(sample - mu) <= 2*sigma)
within_3 = np.mean(np.abs(sample - mu) <= 3*sigma)
print(f"actually within ±1σ: {within_1:.1%}  (rule of thumb: 68%)")
print(f"actually within ±2σ: {within_2:.1%}  (rule of thumb: 95%)")
print(f"actually within ±3σ: {within_3:.1%}  (rule of thumb: 99.7%)")

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Why does the empirical (68-95-99.7) rule only apply to roughly normal distributions, "
    "and what's a distribution-free fallback (mention Chebyshev's inequality by name)?"
)

✏️ **Your turn:** generate a *skewed* sample (try `np.random.exponential(scale=10, size=5000)`)
and check whether the 68-95-99.7 rule still roughly holds. (Spoiler: it won't.)

In [ ]:
# your skewed-distribution experiment here

---
## 6. Covariance

**Definition:** the average *product* of paired deviations — measures the direction of a
linear relationship between two variables.

$$\text{cov}(X,Y) = \frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})$$

If $X$ and $Y$ are both above their means at once (or both below), the product of their
deviations is **positive**. If one is above while the other is below, it's **negative**.
Summing these products literally tallies "how often, and how strongly, do they move together."

In [ ]:
x = [1, 2, 3, 3.2, 4, 5, 6, 6.2, 7, 8]
y = [2, 3.2, 3, 5, 4.2, 6, 5.1, 7, 6.3, 8]

# 1. From scratch
x_mean = sum(x) / len(x)
y_mean = sum(y) / len(y)

products = [(xi - x_mean) * (yi - y_mean) for xi, yi in zip(x, y)]
cov_from_scratch = sum(products) / len(x)

print("x_mean =", x_mean, " y_mean =", y_mean)
print("per-point products:", [round(p, 2) for p in products])
print("covariance:", round(cov_from_scratch, 4))

In [ ]:
# 2. Check with a library
print("numpy (population, via cov matrix):", np.cov(x, y, ddof=0)[0, 1])
print("pandas (sample, ddof=1 by default):", pd.Series(x).cov(pd.Series(y)))
assert math.isclose(cov_from_scratch, np.cov(x, y, ddof=0)[0, 1])

In [ ]:
# 3. A picture — quadrants around the means, with each product drawn as a signed rectangle
fig, ax = plt.subplots(figsize=(6, 6))
ax.axvline(x_mean, color="#c98a4b", ls="--"); ax.axhline(y_mean, color="#c98a4b", ls="--")
for xi, yi in zip(x, y):
    color = "#4a7c9e" if (xi - x_mean) * (yi - y_mean) >= 0 else "#b3728a"
    rect = plt.Rectangle((min(xi, x_mean), min(yi, y_mean)),
                          abs(xi - x_mean), abs(yi - y_mean),
                          fill=False, edgecolor=color, lw=1.3)
    ax.add_patch(rect)
    ax.scatter([xi], [yi], color="#111", zorder=3, s=40)
ax.set_title(f"cov(X,Y) ≈ {cov_from_scratch:.2f}  (blue = positive term, rose = negative term)")
ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Explain the real-world link between covariance and portfolio diversification in "
    "finance — why do investors look for assets with negative covariance?"
)

✏️ **Your turn:** flip the sign of every `y` value (`y2 = [-v for v in y]`) and recompute
covariance from scratch. What happens to the sign, and why?

In [ ]:
# your experiment here

---
## 7. Correlation

**Definition (Pearson's r):** covariance rescaled by both standard deviations, so it's
always between −1 and +1 — a **unit-free** score for how tightly points hug a straight line.

$$r = \frac{\text{cov}(X,Y)}{\sigma_X\,\sigma_Y}$$

The bound isn't a design choice — it's a real theorem. The **Cauchy–Schwarz inequality**
guarantees $|\text{cov}(X,Y)| \le \sigma_X\sigma_Y$ for *any* data, which is exactly why
$r$ can never leave $[-1, 1]$.

In [ ]:
# 1. From scratch (reusing x, y from the covariance section)
x_std = (sum((xi - x_mean) ** 2 for xi in x) / len(x)) ** 0.5
y_std = (sum((yi - y_mean) ** 2 for yi in y) / len(y)) ** 0.5

r_from_scratch = cov_from_scratch / (x_std * y_std)
print(f"σx = {x_std:.4f}, σy = {y_std:.4f}")
print(f"r  = {r_from_scratch:.4f}")

In [ ]:
# 2. Check with a library
print("numpy corrcoef:", np.corrcoef(x, y)[0, 1])
r_scipy, p_scipy = stats.pearsonr(x, y)
print(f"scipy pearsonr: r={r_scipy:.4f}, p-value={p_scipy:.4g}")
assert math.isclose(r_from_scratch, np.corrcoef(x, y)[0, 1])

In [ ]:
# 3. A picture — three relationships, three r values
rng = np.random.default_rng(7)
n = 60
strong_x = rng.uniform(0, 10, n); strong_y = 0.9*strong_x + rng.normal(0, 0.6, n)
none_x   = rng.uniform(0, 10, n); none_y   = rng.normal(5, 2, n)
neg_x    = rng.uniform(0, 10, n); neg_y    = -0.9*neg_x + rng.normal(10, 0.6, n)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=False)
for ax, (xs, ys, title) in zip(
    axes,
    [(strong_x, strong_y, "strong positive"), (none_x, none_y, "~no correlation"), (neg_x, neg_y, "strong negative")]
):
    r = np.corrcoef(xs, ys)[0, 1]
    ax.scatter(xs, ys, s=18, color="#4a7c9e", alpha=0.8)
    ax.set_title(f"{title}\nr = {r:.2f}")
plt.tight_layout(); plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Give one memorable real-world example of correlation without causation, and explain "
    "in plain terms why the correlation doesn't prove one variable causes the other."
)

✏️ **Your turn:** compute `r²` (the coefficient of determination) for the `strong_x`/`strong_y`
pair above, and explain in a comment what fraction of the variance it represents.

In [ ]:
# your r-squared calculation here

---
## 8. Percentiles

**Definition:** the k-th percentile is the value below which k% of the (sorted) data falls.

$$\text{rank} = \left\lceil \frac{k}{100} \times n \right\rceil$$

Quartiles are just special percentiles: Q1=P25, Q2=P50 (the median), Q3=P75. The
**interquartile range**, $\text{IQR} = Q3 - Q1$, is a robust spread measure and the usual
basis for flagging outliers (commonly, anything beyond $1.5\times\text{IQR}$ past Q1 or Q3).

In [ ]:
rng = np.random.default_rng(99)
data = sorted(rng.integers(5, 95, size=20).tolist())
print("sorted data:", data)

# 1. From scratch — nearest-rank method
def percentile_from_scratch(sorted_data, k):
    n = len(sorted_data)
    rank = math.ceil((k / 100) * n)
    rank = max(1, min(rank, n))          # clamp into [1, n]
    return sorted_data[rank - 1]         # -1 for 0-based indexing

for k in [25, 50, 75, 90]:
    print(f"P{k} = {percentile_from_scratch(data, k)}")

In [ ]:
# 2. Check with a library — note: numpy defaults to LINEAR interpolation, a different
#    (also valid) method, so don't be surprised if it disagrees slightly with the
#    nearest-rank version above.
for k in [25, 50, 75, 90]:
    print(f"numpy  P{k}:", np.percentile(data, k))
print("\npandas describe():")
print(pd.Series(data).describe())

In [ ]:
# 3. A picture — sorted data with quartile markers
fig, ax = plt.subplots(figsize=(8, 2.2))
ax.scatter(data, [0]*len(data), color="#4a7c9e", s=60, zorder=3)
for k, color, label in [(25, "#7fa38a", "Q1"), (50, "#c98a4b", "median"), (75, "#7fa38a", "Q3")]:
    val = np.percentile(data, k)
    ax.axvline(val, color=color, ls="--")
    ax.text(val, 0.15, f"{label}\n{val:.0f}", ha="center", fontsize=9)
ax.set_yticks([]); ax.set_title("Sorted data with quartile markers")
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Why do software engineers care about 'p99 latency' instead of just average latency? "
    "Explain with a concrete scenario."
)

✏️ **Your turn:** compute the IQR (`Q3 - Q1`) for `data` above, then flag any values more
than `1.5 * IQR` beyond Q1 or Q3 as outliers.

In [ ]:
# your IQR / outlier-flagging code here

---
## 9. Confidence Interval

**Definition:** a range, built from sample data, likely to contain the true population
parameter.

$$\text{CI} = \bar{x} \pm z^{*}\cdot\frac{\sigma}{\sqrt{n}}$$

$\sigma/\sqrt{n}$ is the **standard error of the mean**, and it follows directly from the
Central Limit Theorem: averaging $n$ independent draws shrinks the *variance* by a factor
of $n$, so the *spread* of the sampling distribution shrinks by $\sqrt{n}$. $z^{*}\approx1.96$
is simply the point on the standard normal curve where 95% of the area falls within
$\pm z^{*}$ of the center.

> "95% confidence" describes the **procedure**, not one specific interval: over many
> repeated samples, about 95% of the intervals built this way would contain the true value.

In [ ]:
true_mu, true_sigma = 50, 8     # the (usually unknown) population truth
n = 30

sample = np.random.normal(true_mu, true_sigma, n)
x_bar = sample.mean()

# 1. From scratch
z_star = 1.96                                  # for 95% confidence
standard_error = true_sigma / math.sqrt(n)
margin = z_star * standard_error
ci_low, ci_high = x_bar - margin, x_bar + margin

print(f"sample mean       = {x_bar:.2f}")
print(f"standard error     = {standard_error:.3f}")
print(f"95% CI (from scratch) = [{ci_low:.2f}, {ci_high:.2f}]")
print(f"does it contain the true mean ({true_mu})? {ci_low <= true_mu <= ci_high}")

In [ ]:
# 2. Check with a library
ci_scipy = stats.norm.interval(0.95, loc=x_bar, scale=standard_error)
print("scipy 95% CI:", tuple(round(v, 2) for v in ci_scipy))
# note: 1.96 is a rounded approximation of the true 97.5th-percentile z-value
# (which is closer to 1.9599...), so we compare with a loose tolerance, not exact equality.
assert math.isclose(ci_low, ci_scipy[0], rel_tol=1e-3)
print("more precise z* from scipy:", round(stats.norm.ppf(0.975), 6))

In [ ]:
# 3. A picture — simulate 20 experiments, see how many of the 20 intervals catch the truth
num_experiments = 20
captured = 0

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvline(true_mu, color="black", lw=1.5, label="true mean (usually unknown)")

for i in range(num_experiments):
    s = np.random.normal(true_mu, true_sigma, n)
    xb = s.mean()
    se = true_sigma / math.sqrt(n)
    lo, hi = xb - 1.96*se, xb + 1.96*se
    hit = lo <= true_mu <= hi
    captured += hit
    ax.plot([lo, hi], [i, i], color="#4a7c9e" if hit else "#b3728a", lw=2)
    ax.scatter([xb], [i], color="black", s=10, zorder=3)

ax.set_yticks([]); ax.legend()
ax.set_title(f"{captured}/{num_experiments} intervals captured the true mean "
             f"({captured/num_experiments:.0%}, theory says ~95%)")
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Correct this common misconception in plain language: 'there's a 95% probability the "
    "true mean is inside this one confidence interval.' What's actually true instead?"
)

✏️ **Your turn:** change `n` to 200 (much bigger sample) and re-run the simulation cell.
Notice the intervals get narrower — that's the $\sqrt{n}$ in the denominator at work.

In [ ]:
# re-run the simulation with a larger n here

---
## 10. Hypothesis Testing

**Framework:** assume a "null" scenario $H_0$ (nothing interesting is happening), then ask
whether the observed data is surprising enough, under that assumption, to doubt it.

$$z = \frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$$

Standardizing to $z$ expresses "how many standard errors away from the null's prediction is
my data" as one number — comparable against a universal reference table, regardless of the
original units. Reject $H_0$ if $|z|$ exceeds a critical value tied to your chosen
significance level $\alpha$ (commonly 0.05, giving $z_{crit}\approx1.96$ two-tailed).

In [ ]:
# Scenario: a factory's bolts have historically had mean diameter 10.00mm, sigma 0.5mm.
# A new batch of 40 bolts is measured; is the new process producing a different average?
mu_0 = 10.00
sigma = 0.50
n = 40
alpha = 0.05

rng = np.random.default_rng(3)
sample = rng.normal(10.18, sigma, n)   # the new batch, secretly shifted for this demo
x_bar = sample.mean()

# 1. From scratch
standard_error = sigma / math.sqrt(n)
z = (x_bar - mu_0) / standard_error
z_crit = 1.96                                    # two-tailed, alpha = 0.05

print(f"sample mean x̄ = {x_bar:.4f}")
print(f"z = ({x_bar:.4f} - {mu_0}) / {standard_error:.4f} = {z:.3f}")
print(f"|z| = {abs(z):.3f}  vs.  z_crit = {z_crit}")
print("=> REJECT H0" if abs(z) > z_crit else "=> FAIL TO REJECT H0")

In [ ]:
# 2. Check with a library — a one-sample z-test isn't built into scipy directly (scipy's
#    ttest_1samp is the more common t-test cousin), so we verify by hand-computing the
#    two-tailed p-value from the standard normal and comparing to alpha.
p_value_check = 2 * (1 - stats.norm.cdf(abs(z)))
print("p-value from scipy's normal CDF:", round(p_value_check, 5))
print("reject H0 at alpha=0.05?", p_value_check < alpha)

In [ ]:
# 3. A picture — the null distribution, critical region, and where our statistic landed
xs = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, stats.norm.pdf(xs), color="#111")
ax.fill_between(xs, stats.norm.pdf(xs), where=(xs > z_crit) | (xs < -z_crit),
                 color="#b3728a", alpha=0.35, label="rejection region (α=0.05)")
ax.axvline(z, color="#c98a4b", lw=2, label=f"observed z = {z:.2f}")
ax.legend(); ax.set_title("H0: μ = 10.00mm  —  standardized test statistic z")
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "Explain the difference between a Type I error and a Type II error using the bolt "
    "factory example: mu_0=10.00mm, and a new batch that might have shifted to 10.18mm."
)

✏️ **Your turn:** change `mu_0` to `10.18` (i.e. assume the null hypothesis matches the true
shifted mean) and re-run. You should now fail to reject H0 far more often — that's exactly
what should happen when the null hypothesis is actually true.

In [ ]:
# your experiment here

---
## 11. p-value

**Definition:** $p = P(\text{data this extreme or more} \mid H_0 \text{ true})$ — how
surprising the data would be, if the null hypothesis were actually correct.

A p-value is **not** the probability that $H_0$ is true, and it's **not** the probability the
result happened "by chance" — two of the most common misreadings in all of statistics.

In [ ]:
# Reusing the z-statistic from the hypothesis-testing section
print(f"z = {z:.3f}")

# 1. From scratch — using the standard normal survival function's mathematical form.
#    (We lean on scipy's norm.cdf here for the actual integral — nobody hand-integrates
#    a Gaussian — but the p-value LOGIC below is fully spelled out.)
one_tailed_p = 1 - stats.norm.cdf(abs(z))     # P(Z >= |z|)
two_tailed_p = 2 * one_tailed_p               # "or more extreme" in EITHER direction

print(f"one-tailed p-value: {one_tailed_p:.5f}")
print(f"two-tailed p-value: {two_tailed_p:.5f}")

In [ ]:
# 2. Check with a library
print("scipy norm.sf (survival function, i.e. 1-cdf):", stats.norm.sf(abs(z)) * 2)
assert math.isclose(two_tailed_p, stats.norm.sf(abs(z)) * 2, rel_tol=1e-6)

In [ ]:
# 3. A picture — the shaded tail area IS the p-value
xs = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, stats.norm.pdf(xs), color="#111")
ax.fill_between(xs, stats.norm.pdf(xs), where=(xs >= abs(z)), color="#c98a4b", alpha=0.5)
ax.fill_between(xs, stats.norm.pdf(xs), where=(xs <= -abs(z)), color="#c98a4b", alpha=0.5)
ax.axvline(z, color="#4a7c9e", lw=2, label=f"observed z = {z:.2f}")
ax.set_title(f"two-tailed p-value ≈ {two_tailed_p:.4f}  (shaded area)")
ax.legend()
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "In plain language, what's wrong with the statement 'a p-value of 0.03 means there's "
    "a 3% chance the null hypothesis is true'? What should someone say instead?"
)

✏️ **Your turn:** with a much larger sample size (`n = 4000` instead of 40, same tiny shift
in the mean), recompute the p-value. Notice it can become tiny even for a practically
meaningless effect size — that's the statistical-significance-vs-practical-significance trap.

In [ ]:
# your large-n experiment here

---
## 12. ANOVA

**Definition:** Analysis of Variance — tests whether three or more group means are likely
all equal, using a single test instead of many pairwise t-tests.

$$F = \frac{MS_{between}}{MS_{within}} = \frac{SS_{between}/(k-1)}{SS_{within}/(N-k)}$$

If the group means truly differ, the "between-group" variance (how spread out the group
means are) should be large relative to the "within-group" variance (ordinary noise inside
each group) — the F-ratio is precisely engineered to compare that signal to that noise.

In [ ]:
# Scenario: crop yield (kg) under three different fertilizers
rng = np.random.default_rng(3)
group_A = rng.normal(4.2, 1.1, 12).round(2)
group_B = rng.normal(5.6, 1.1, 12).round(2)
group_C = rng.normal(7.1, 1.1, 12).round(2)
groups = [group_A, group_B, group_C]
k = len(groups)                      # number of groups
N = sum(len(g) for g in groups)      # total observations

for name, g in zip("ABC", groups):
    print(f"group {name}: mean={g.mean():.2f}, n={len(g)}")

In [ ]:
# 1. From scratch
grand_mean = np.concatenate(groups).mean()

ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_within  = sum(((g - g.mean()) ** 2).sum() for g in groups)

df_between = k - 1
df_within  = N - k

ms_between = ss_between / df_between
ms_within  = ss_within / df_within

F_from_scratch = ms_between / ms_within

print(f"grand mean       = {grand_mean:.3f}")
print(f"SS_between = {ss_between:.3f}  (df={df_between})  ->  MS_between = {ms_between:.3f}")
print(f"SS_within  = {ss_within:.3f}  (df={df_within})  ->  MS_within  = {ms_within:.3f}")
print(f"F = {F_from_scratch:.3f}")

In [ ]:
# 2. Check with a library
F_scipy, p_scipy = stats.f_oneway(*groups)
print(f"scipy f_oneway: F={F_scipy:.3f}, p={p_scipy:.5f}")
assert math.isclose(F_from_scratch, F_scipy, rel_tol=1e-6)

print("\nConclusion:", "REJECT H0 — groups likely differ" if p_scipy < 0.05
                          else "FAIL TO REJECT H0 — no strong evidence groups differ")

In [ ]:
# 3. A picture — the groups, their means, and the grand mean
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#4a7c9e", "#7fa38a", "#c98a4b"]
for i, (name, g, color) in enumerate(zip("ABC", groups, colors)):
    jitter = rng.uniform(-0.12, 0.12, len(g))
    ax.scatter(np.full(len(g), i) + jitter, g, color=color, alpha=0.85, s=45, label=f"group {name}")
    ax.hlines(g.mean(), i - 0.25, i + 0.25, color=color, lw=3)
ax.axhline(grand_mean, color="black", ls="--", label="grand mean")
ax.set_xticks(range(k)); ax.set_xticklabels(["A", "B", "C"])
ax.set_title(f"F = {F_from_scratch:.2f},  p = {p_scipy:.4f}")
ax.legend()
plt.show()

🤖 **Ask the AI tutor**

In [ ]:
ask_ai(
    "If ANOVA on three fertilizer groups gives a significant F, why can't we conclude "
    "which specific fertilizer is best just from that F-statistic? What's the next step?"
)

✏️ **Your turn:** run `scipy.stats.tukey_hsd(*groups)` (available in recent scipy versions)
to see the post-hoc pairwise comparisons and find out exactly which groups differ.

In [ ]:
# your post-hoc test here
# from scipy.stats import tukey_hsd
# result = tukey_hsd(*groups)
# print(result)

---
## 🤖 Ask the AI tutor anything

Every concept above stands on its own without the AI — but now that you've seen the pattern,
use this cell to ask about anything that's still fuzzy. Try things like:

- *"Walk me through when I'd use a t-test instead of a z-test."*
- *"How is standard error different from standard deviation?"*
- *"Give me a made-up dataset and quiz me on computing its mean, median, and mode."*

In [ ]:
your_question = "How is standard error different from standard deviation?"
ask_ai(your_question)

---
### Where to go next

- Swap every toy dataset above for one of your own — the from-scratch code will still work unchanged.
- Try `statsmodels` for a deeper dive into regression, ANOVA tables, and hypothesis testing.
- Pair this notebook with the companion animated HTML deck for the visual intuition behind
  each formula before diving into the code.